<a href="https://colab.research.google.com/github/FranciscoBPereira/Bioinformatica26-27/blob/main/Bioinfo2627_Aula8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**0. Instalar Biopython**
https://biopython.org/


In [ ]:
!pip install BioPython

from Bio.PDB import *


**Tarefa 1: Acesso à PDB para obtenção de um ficheiro cif**

**1.1 Obter informação sobre a estrutura 3D de uma proteína**

In [ ]:
# Obter o ficheiro cif a partir da base de dados PDB
# O identificador PDB da proteína deve ser indicado pelo utilizador

# O ficheiro cif fica armazenado numa pasta na diretoria corrente
idP = "2K39"  #@param {type:"string"}

pdbl = PDBList()
caminho = pdbl.retrieve_pdb_file(idP, file_format='mmCif')



**1.2 Criar e trabalhar com um objeto Structure**

In [ ]:
# Criar um objeto Structure a partir da informação do ficheiro cif (ignorar os warnings)

parser = MMCIFParser()

estrutura = parser.get_structure(idP, caminho)

In [ ]:
# Verifica o modo experimental usado para calcular a estrutura tridimensional


print('Metodo: ', estrutura.header['structure_method'])


In [ ]:
# Verificar quantos modelos existem no ficheiro

for modelo in estrutura:
  print('Modelo:',modelo.id)

**Questão:**

Explique o motivo pelo qual existem vários modelos

In [ ]:
# Selecionar apenas o primeiro modelo
# Verificar quantas cadeias tem este modelo

model1 = estrutura[0]
for cadeia in model1:
  print('Cadeia:',cadeia.id)

In [ ]:
# Apresentar informação mais detalhada do modelo
# Cadeias
# Resíduos por cadeia


for cadeia in model1:
  print('Cadeia:',cadeia.id)
  for residuo in cadeia:
      print('\t\tResiduo:',residuo.resname,"(",residuo.id[1],")")


In [ ]:
# Apresentar informação completa do modelo:
# Cadeias
# Resíduos por cadeia
# Átomos por resíduo, com localização

for cadeia in model1:
  print('Cadeia:',cadeia.id)
  for residuo in cadeia:
      print('\t\tResiduo:',residuo.resname,"(",residuo.id[1],")")
      for atomo in residuo:
        print("\t\t\tAtomo:",atomo.name,"\
        \tCoordenadas: (X:",atomo.coord[0],\
        "L Y:",atomo.coord[1],\
        "L Z:",atomo.coord[2],")")

In [ ]:
# Contar o número de átomos por cadeia

for cadeia in model1:
  print('Cadeia:',cadeia.id)
  num_atomos = 0
  for residuo in cadeia:
      for atomo in residuo:
        num_atomos += 1
  print('\t\tNúmero de átomos:',num_atomos)



In [ ]:
# Contar o número de átomos por resíduo


for cadeia in model1:
  print('Cadeia:',cadeia.id)
  for residuo in cadeia:
      print('\t\tResiduo:',residuo.resname,"(",residuo.id[1],")")
      num_atomos = 0
      for atomo in residuo:
        num_atomos += 1
      print('\t\t\tNúmero de átomos:',num_atomos)

In [ ]:
# Apresentar uma figura 3D com a localização de todos os átomos deste modelo

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(16,9))
ax3d=fig.add_subplot(111,projection='3d')

for atomo in model1.get_atoms():
  ax3d.scatter(atomo.coord[0],atomo.coord[1],atomo.coord[2])

ax3d.set_xlabel('X')
ax3d.set_ylabel('Y')
ax3d.set_zlabel('Z')
plt.show()

In [ ]:
# Gráfico de Ramachandran: Apresenta as combinações ângulos da cadeia principal da proteína
# Avalia a qualidade geométrica dos modelos tridiomensionais


import math
import matplotlib.pyplot as plt
from Bio.PDB import PPBuilder

classes = {"outros": ([], []), "GLY": ([], []), "PRO": ([], []), "pré-PRO": ([], [])}


title = "Ramachandran - " +  idP

for pp in PPBuilder().build_peptides(model1):
    angulos = pp.get_phi_psi_list()
    for i, (res, (f, p)) in enumerate(zip(pp, angulos)):
        if f is None or p is None:
            continue
        nome = res.get_resname()
        if nome in ("GLY", "PRO"):
            chave = nome
        elif i + 1 < len(pp) and pp[i + 1].get_resname() == "PRO":
            chave = "pré-PRO"
        else:
            chave = "outros"
        classes[chave][0].append(math.degrees(f))
        classes[chave][1].append(math.degrees(p))

estilo = {"outros": "tab:blue", "GLY": "tab:orange", "PRO": "tab:green", "pré-PRO": "tab:red"}

fig, ax = plt.subplots(figsize=(6, 6))
for nome in ("outros", "GLY", "PRO", "pré-PRO"):   # classes minoritárias desenhadas por cima
    x, y = classes[nome]
    ax.scatter(x, y, s=8, c=estilo[nome], label=f"{nome} (n={len(x)})")

ax.set(xlim=(-180, 180), ylim=(-180, 180), aspect="equal",
       xticks=range(-180, 181, 60), yticks=range(-180, 181, 60),
       xlabel="φ (graus)", ylabel="ψ (graus)")
ax.set_title(title)
ax.axhline(0, c="grey", lw=0.5)
ax.axvline(0, c="grey", lw=0.5)
ax.legend(loc="lower left", fontsize=8)
fig.savefig("ramachandran.png", dpi=150, bbox_inches="tight")